In [ ]:
# from google.colab import drive

# drive.mount('/content/drive')

# %cd /content/drive/MyDrive/faster_rcnn
# %cp VOC2007.zip /content
# %cp VOC2012.zip /content

In [ ]:
from pathlib import Path
import zipfile

data_path = Path("data/")
data_path.mkdir(exist_ok=True)

voc2007_zip_path = Path("VOC2007.zip")
voc2012_zip_path = Path("VOC2012.zip")

if not voc2007_zip_path.exists() or not voc2012_zip_path.exists():
    raise RuntimeError("Dataset not found.")

print("Extracting 2007 dataset ...")

with zipfile.ZipFile(voc2007_zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

print(f"Extracting 2012 dataset ...")
with zipfile.ZipFile(voc2012_zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from src.backbone import backbone_transform
from src.dataset import get_voc_img_paths_train, get_voc_img_paths_test, create_voc_dataloader

transform = backbone_transform

voc2007_img_paths_train, voc2012_img_paths_train = get_voc_img_paths_train()
voc2007_img_paths_test, voc2012_img_paths_test = get_voc_img_paths_test()

combined_train_img_paths = voc2007_img_paths_train + voc2012_img_paths_train

train_dataloader = create_voc_dataloader(img_paths_list=combined_train_img_paths, transform=transform, batch_size=2, shuffle=True)
test_dataloader = create_voc_dataloader(img_paths_list=voc2007_img_paths_test, transform=transform, batch_size=2, shuffle=True)

In [ ]:
from src.backbone import Backbone
from src.rpn import RPN_Head, RegionProposalNetwork
from src.roi import RoIPool
from detection_net import DetectionHead, DetectionNet
import torch
from pathlib import Path

# Step-1 RPN network: frozen, exists only to generate proposals.
backbone_rpn = Backbone().to(device)
rpn_head = RPN_Head(in_channels=1024, mid_channels=512)

# rpn_checkpoint_path = Path("/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_10.pt")
rpn_checkpoint_path = Path("checkpoints/step1_epoch_10.pt")
rpn_checkpoint = torch.load(rpn_checkpoint_path, map_location=device)

backbone_rpn.load_state_dict(rpn_checkpoint['backbone_state_dict'])
rpn_head.load_state_dict(rpn_checkpoint['rpn_head_state_dict'])
print(f"Loaded RPN Checkpoint: {rpn_checkpoint_path}")

rpn_net = RegionProposalNetwork(rpn_head).to(device)

for param in backbone_rpn.parameters():
    param.requires_grad = False
for param in rpn_head.parameters():
    param.requires_grad = False

# eval() here and never train() again
backbone_rpn.eval()
rpn_net.eval()

# Step-2 detection network: separate, fresh ImageNet weights, trained.
backbone = Backbone().to(device)
roi_pool = RoIPool(output_size=(7, 7), pooling_mode="adaptive").to(device)
detection_head = DetectionHead().to(device)


# Fn to Freeze BatchNorm layers in the backbone to avoid updating running stats during training conv_5x layers BN was already frozen during detection_head creation
def freeze_batchnorm(module):
    for m in module.modules():
        if isinstance(m, torch.nn.BatchNorm2d):
            m.eval()
            m.weight.requires_grad = False
            m.bias.requires_grad = False

In [ ]:
from pathlib import Path

# checkpoint_dir = Path("/content/drive/MyDrive/faster_rcnn/checkpoints")
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

existing_checkpoints = sorted(checkpoint_dir.glob("step2_epoch_*.pt"),
                              key = lambda p : int(p.stem.split("_epoch_")[1]))

if existing_checkpoints:
    latest_checkpoint = existing_checkpoints[-1]
    print(f"Loading checkpoint: {latest_checkpoint}")

    checkpoint = torch.load(latest_checkpoint, map_location=device)
    
    backbone.load_state_dict(checkpoint['backbone_state_dict'])
    detection_head.load_state_dict(checkpoint['detection_head_state_dict'])

else:
    print("No existing checkpoints found. Starting training from scratch.")


In [ ]:
batch_imgs, batch_boxes, batch_labels, batch_img_sizes_before_pad = next(iter(test_dataloader))

rpn_network = RegionProposalNetwork(rpn_head=rpn_head).to(device)
detection_network = DetectionNet(detection_head=detection_head, background_label=20, delta_std=(0.1, 0.1, 0.2, 0.2)).to(device)

with torch.inference_mode():
    backbone_rpn.eval()
    rpn_head.eval()
    rpn_network.eval()

    backbone.eval()
    roi_pool.eval()
    detection_head.eval()
    detection_network.eval()

    rpn_feature_maps = backbone_rpn(batch_imgs)
    batch_scores, batch_proposals = rpn_net(rpn_feature_maps, batch_img_height=batch_imgs.shape[2],  batch_img_width=batch_imgs.shape[3], img_sizes_before_pad=batch_img_sizes_before_pad, pre_nms_top_n=6000, post_nms_top_n=2000)
    

    batch_feature_maps = backbone(batch_imgs.to(device))
    print(f"batch_feature_maps.shape: {batch_feature_maps.shape}")
    pooled_batch = roi_pool(batch_feature_maps, batch_proposals, batch_imgs.shape[2], batch_imgs.shape[3])

    batch_pred_labels, batch_pred_scores, batch_pred_boxes = detection_network(batch_proposals, pooled_batch, batch_img_sizes_before_pad)


In [ ]:
from src.utils import display_random_batch_images

display_random_batch_images(batch_imgs, batch_boxes, batch_labels, batch_scores, img_sizes_before_pad, num_images=4)
display_random_batch_images(batch_imgs, batch_boxes, batch_labels, batch_scores, img_sizes_before_pad, num_images=4)